In [ ]:
# ! pip install -e ../../savo

In [ ]:
import numpy as np
import torch
import matplotlib.pyplot as plt
import time
import sys
sys.path.append('../')
from machineIO.objFunc import SingleTaskObjectiveFunction
from machineIO import Evaluator, OracleEvaluator
from IPython.display import display

In [ ]:
n_input = 4
n_output = 2


control_CSETs = [f'X{i}:I_CSET' for i in range(1,n_input+1)]
control_RDs   = [f'X{i}:I_RD' for i in range(1,n_input+1)]
control_min   = -np.ones(n_input)
control_max   =  np.ones(n_input)
control_tols  = 1e-3*(control_max-control_min)
control_init = np.random.randn(n_input)*(control_max - control_min) + control_min
monitor_PVs   = [f'Y{i}' for i in range(1,n_output+1)]
monitor_min   = -2*np.ones(n_output)
monitor_max   =  2*np.ones(n_output)

objective_PVs = monitor_PVs
composite_objective_name = 'rastrigin'

# machineIO

In [ ]:
from epics import caget
from machineIO import construct_machineIO

t0 = time.time()
io = construct_machineIO(use_epics=True)
t1 = time.time()
print("--------")
print(f"{int(t1-t0)} sec")

# read using epics
val = [caget("X1:I_CSET"),caget("X1:I_RD")]
t2 = time.time()
print(val)
print(f"{int(t2-t1)} sec")

# read using io
val = [io.caget("X1:I_CSET"),io.caget("X1:I_RD")]
t3 = time.time()
print(val)
print(f"{int(t3-t2)} sec")


# put using io
io.caput("X1:I_CSET",1)
t4 = time.time()
print(f"{int(t4-t3)} sec")


# check result 
time.sleep(0.1) # wait for nework to update CSET PV value
val = [io.caget("X1:I_CSET"),io.caget("X1:I_RD")]
t5 = time.time()
print(val)
print(f"{int(t5-t4)} sec")


# wait and check
time.sleep(5)
t5 = time.time()
val = [io.caget("X1:I_CSET"),io.caget("X1:I_RD")]
t6 = time.time()
print(val)
print(f"{int(t6-t5)} sec")


In [ ]:
# Ramp and verify (returns ret string + ramping df)

df = io.fetch_data(["X1:I_CSET", "X1:I_RD"], time_span=2.0, sample_interval=0.2)
display(df)

ret, ramp_df = io.ensure_set(
    setpoint_pv=["X1:I_CSET", "X2:I_CSET"],
    readback_pv=["X1:I_RD", "X2:I_RD"],
    goal=[2.0, 2.0],
    tol=[0.1, 0.1],
    timeout=20,
    sample_interval=0.2,
    extra_monitors=["Y1"],
)
print(ret)
display(ramp_df)


In [ ]:
ev = Evaluator(
    machineIO=io,
    control_CSETs=control_CSETs,
    control_RDs=control_RDs,
    control_tols=control_tols,
    monitor_PVs=monitor_PVs,
)
future = ev.submit([1]*n_input)
df, ramp_df = ev.get_result(future)

In [ ]:
ramp_df

In [ ]:
df

# obj_func

: rastirigin over 2D latent space of random NN for both high-dim and visualization

In [ ]:
noise = 0.0

def rastrigin(x,noise=noise):
    x = torch.as_tensor(x)
    b,d = x.shape
    y = torch.sum(x**2 - torch.cos(2*np.pi*x),axis=1)/d +1
    return 1-y + torch.randn(b)*noise

In [ ]:
grid = np.linspace(-2,2,128)
x1,x2 = np.meshgrid(grid,grid)
xgrid = np.vstack((x1.flatten(), x2.flatten())).T
ygrid = rastrigin(xgrid,noise=0)
def plot_contour(figsize=(4,3.3),dpi=128):
    fig,ax = plt.subplots(figsize=figsize,dpi=dpi)
    cs = ax.tricontourf(xgrid[:,0],xgrid[:,1],ygrid,levels=32)
    fig.colorbar(cs,ax=ax)
    return fig,ax
fig,ax = plot_contour(dpi=32)

In [ ]:
obj_func = SingleTaskObjectiveFunction(
    objective_PVs = monitor_PVs,
    composite_objective_name = composite_objective_name,
    custom_function = rastrigin,
    objective_goal = None, 
    objective_weight = None,
    objective_tolerance = None,
)

In [ ]:
evaluator = Evaluator(
    machineIO = io,
    control_CSETs = control_CSETs,
    control_RDs = control_RDs,
    control_tols = control_tols,
    monitor_PVs = monitor_PVs,
    df_manipulators = [obj_func.calculate_objectives_from_df],
)

In [ ]:
oracle_key_names = {f'x{i+1}':f'X{i+1}:I_RD' for i in range(len(control_CSETs))}
oracle_key_names['y'] = 'rastrigin'

oracle = OracleEvaluator(
    machineIO = io,
    control_CSETs = control_CSETs,
    control_RDs = control_RDs,
    control_tols = control_tols,
    monitor_PVs = monitor_PVs,
    oracle_key_names = oracle_key_names,
    df_manipulators = [obj_func.calculate_objectives_from_df],

)

In [ ]:
oracle_dic = oracle()
oracle_dic

In [ ]:
oracle_dic = oracle(np.zeros(len(control_CSETs)))
oracle_dic